# Train TRM for 2048

This notebook trains a Tiny Recursive Model (TRM) to play 2048 using Expectimax-generated training data.

**Steps:**
1. Imports & setup
2. Generate training data (Expectimax teacher)
3. Train TRM with loss plot
4. Evaluate on validation set
5. Save checkpoint

In [ ]:
# Cell 1: Imports & setup
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from src.data.game2048 import generate_training_data, Game2048Dataset, VOCAB_SIZE, DIRS

print(f'PyTorch {torch.__version__}')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

In [ ]:
# Cell 2: Generate training data
N_GAMES = 500  # increase to 5000 for full training
print(f'Generating {N_GAMES} games with Expectimax(depth=2) teacher...')
data = generate_training_data(n_games=N_GAMES, depth=2)
print(f'Total (board, label) pairs: {len(data)}')

random.shuffle(data)
n_val = max(1, int(0.1 * len(data)))
train_ds = Game2048Dataset(data[n_val:])
val_ds   = Game2048Dataset(data[:n_val])
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=256)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

In [ ]:
# Cell 3: TRM model + training loop with loss plot

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.w = nn.Parameter(torch.ones(d))
    def forward(self, x):
        return x / torch.sqrt(torch.mean(x**2, dim=-1, keepdim=True) + self.eps) * self.w

class SwiGLU(nn.Module):
    def __init__(self, d):
        super().__init__()
        h = int(8/3 * d)
        self.w1 = nn.Linear(d, h, bias=False)
        self.w2 = nn.Linear(h, d, bias=False)
        self.w3 = nn.Linear(d, h, bias=False)
    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

class Attn(nn.Module):
    def __init__(self, d, h=4):
        super().__init__()
        self.h, self.dk = h, d // h
        self.qkv = nn.Linear(d, 3*d, bias=False)
        self.o   = nn.Linear(d, d,   bias=False)
    def forward(self, x):
        B, L, D = x.shape
        q, k, v = self.qkv(x).reshape(B, L, 3, self.h, self.dk).permute(2, 0, 3, 1, 4)
        attn = F.softmax(q @ k.transpose(-2,-1) / (self.dk**0.5), dim=-1)
        return self.o((attn @ v).transpose(1,2).reshape(B, L, D))

class Block(nn.Module):
    def __init__(self, d, h=4):
        super().__init__()
        self.n1, self.a = RMSNorm(d), Attn(d, h)
        self.n2, self.f = RMSNorm(d), SwiGLU(d)
    def forward(self, x):
        x = x + self.a(self.n1(x))
        return x + self.f(self.n2(x))

class Net(nn.Module):
    def __init__(self, d=128, l=2, h=4):
        super().__init__()
        self.ls = nn.ModuleList([Block(d, h) for _ in range(l)])
        self.n  = RMSNorm(d)
    def forward(self, x):
        for layer in self.ls: x = layer(x)
        return self.n(x)

class LatentRec(nn.Module):
    def __init__(self, net, d=128, n=4):
        super().__init__()
        self.net, self.n = net, n
        self.zp = nn.Linear(3*d, d, bias=False)
        self.yp = nn.Linear(2*d, d, bias=False)
    def forward(self, x, y, z):
        for _ in range(self.n):
            z = self.net(self.zp(torch.cat([x, y, z], dim=-1)))
        return self.net(self.yp(torch.cat([y, z], dim=-1))), z

class TRM(nn.Module):
    def __init__(self, d=128, l=2, h=4, n=4, T=3, v=4):
        super().__init__()
        self.T = T
        net = Net(d, l, h)
        self.rec  = LatentRec(net, d, n)
        self.head = nn.Linear(d, v, bias=False)
    def forward(self, x, y, z):
        with torch.no_grad():
            for _ in range(self.T - 1):
                y, z = self.rec(x, y, z)
        y, z = self.rec(x, y, z)
        return (y, z), self.head(y)

# Instantiate
cfg   = dict(d=128, l=2, h=4, n=4, T=3, v=4)
model = TRM(**cfg)
emb   = nn.Embedding(VOCAB_SIZE, cfg['d'])
opt   = torch.optim.Adam(list(model.parameters()) + list(emb.parameters()), lr=1e-3)

EPOCHS = 10
train_losses, val_accs = [], []

for epoch in range(1, EPOCHS + 1):
    model.train(); emb.train()
    total_loss, n_total = 0.0, 0
    for batch in train_loader:
        tokens, labels = batch['tokens'], batch['label']
        x = emb(tokens)
        y = torch.zeros_like(x); z = torch.zeros_like(x)
        (_, _), y_hat = model(x, y, z)
        logits = y_hat.mean(dim=1)
        loss   = F.cross_entropy(logits, labels)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(list(model.parameters()) + list(emb.parameters()), 1.0)
        opt.step()
        total_loss += loss.item() * labels.size(0)
        n_total    += labels.size(0)
    train_losses.append(total_loss / n_total)

    model.eval(); emb.eval()
    vc, vt = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            tokens, labels = batch['tokens'], batch['label']
            x = emb(tokens)
            y = torch.zeros_like(x); z = torch.zeros_like(x)
            (_, _), y_hat = model(x, y, z)
            preds = y_hat.mean(dim=1).argmax(dim=1)
            vc += (preds == labels).sum().item()
            vt += labels.size(0)
    val_accs.append(vc / vt)
    print(f'Epoch {epoch:2d}/{EPOCHS} | loss {train_losses[-1]:.4f} | val_acc {val_accs[-1]:.3f}')

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(range(1, EPOCHS+1), train_losses, 'b-o', markersize=4)
ax1.set_title('Training Loss'); ax1.set_xlabel('Epoch'); ax1.grid(True, alpha=0.3)
ax2.plot(range(1, EPOCHS+1), val_accs, 'g-o', markersize=4)
ax2.set_title('Validation Accuracy'); ax2.set_xlabel('Epoch'); ax2.set_ylim(0, 1); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 4: Evaluate — confusion over directions
from collections import Counter
correct_per_dir = Counter()
total_per_dir   = Counter()
model.eval(); emb.eval()
with torch.no_grad():
    for batch in val_loader:
        tokens, labels = batch['tokens'], batch['label']
        x = emb(tokens)
        y = torch.zeros_like(x); z = torch.zeros_like(x)
        (_, _), y_hat = model(x, y, z)
        preds = y_hat.mean(dim=1).argmax(dim=1)
        for p, l in zip(preds, labels):
            lbl = DIRS[l.item()]
            total_per_dir[lbl]   += 1
            if p == l:
                correct_per_dir[lbl] += 1

print('Per-direction accuracy:')
for d in DIRS:
    t = total_per_dir[d]
    c = correct_per_dir[d]
    print(f'  {d:8s}: {c}/{t}  ({c/t:.1%})')
print(f'\nOverall val acc: {val_accs[-1]:.3f}')

In [ ]:
# Cell 5: Save checkpoint
import os
os.makedirs('../outputs', exist_ok=True)
best_acc = max(val_accs)
torch.save({
    'model':  model.state_dict(),
    'emb':    emb.state_dict(),
    'acc':    best_acc,
    'config': cfg,
}, '../outputs/2048_trm.pt')
print(f'Saved ../outputs/2048_trm.pt  (best val acc: {best_acc:.3f})')